In [1]:
# =============================================================================
# 🧹 NETTOYAGE SYSTÈME
# =============================================================================
import os, gc, shutil, glob

def cleanup():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except: pass
    
    for pattern in ['**/__pycache__', '**/.ipynb_checkpoints']:
        for p in glob.glob(pattern, recursive=True):
            try: shutil.rmtree(p)
            except: pass
    
    total, used, free = shutil.disk_usage('/')
    print(f"💾 Espace disque: {free/1e9:.1f} GB libre")

cleanup()

💾 Espace disque: 3.0 GB libre


In [2]:
# =============================================================================
# 📦 IMPORTS
# =============================================================================
import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import os
import gc
from typing import List, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
from transformers import CamembertTokenizer, CamembertModel
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA RTX A5000


In [3]:
# =============================================================================
# ⚙️ CONFIGURATION
# =============================================================================
@dataclass
class Config:
    # Paths
    data_dir: str = "./data"
    embedding_dir: str = "./data/embeddings"
    
    # Model
    model_name: str = "camembert-base"
    max_length: int = 128
    batch_size: int = 32
    
    # Embeddings multi-layer
    use_multi_layer: bool = True
    layers_to_use: Tuple[int, ...] = (-1, -2, -3, -4)  # 4 dernières couches
    pooling: str = "mean"  # "cls" ou "mean"
    
    # Mixed precision
    use_fp16: bool = True
    
    def __post_init__(self):
        os.makedirs(self.data_dir, exist_ok=True)
        os.makedirs(self.embedding_dir, exist_ok=True)

config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️ Device: {device}")
print(f"📊 Multi-layer: {config.use_multi_layer} (couches {config.layers_to_use})")
print(f"📊 Pooling: {config.pooling}")


🖥️ Device: cuda
📊 Multi-layer: True (couches (-1, -2, -3, -4))
📊 Pooling: mean


## 1. Chargement des données

In [4]:
# =============================================================================
# 📥 CHARGEMENT DES DONNÉES BRUTES
# =============================================================================

def load_jsonl(filepath: str) -> pd.DataFrame:
    """Charge un fichier JSONL."""
    print(f"Loading {filepath}...")
    df = pd.read_json(filepath, lines=True)
    df = json_normalize(df.to_dict(orient="records"))
    print(f"   → {len(df)} samples, {len(df.columns)} columns")
    return df

def extract_text(row: pd.Series) -> str:
    """Extrait le texte complet du tweet."""
    extended = row.get("extended_tweet.full_text", None)
    if pd.notna(extended) and extended:
        return str(extended)
    text = row.get("text", "")
    return str(text) if pd.notna(text) else ""

# Charger
train_df = load_jsonl("data/train.jsonl")
kaggle_df = load_jsonl("data/kaggle_test.jsonl")

# Extraire labels et textes
y_train = train_df["label"].values
train_texts = train_df.apply(extract_text, axis=1).tolist()
kaggle_texts = kaggle_df.apply(extract_text, axis=1).tolist()

# Sauvegarder labels
np.save(os.path.join(config.data_dir, "y_train.npy"), y_train)

print(f"\n📊 Résumé:")
print(f"   Train: {len(train_texts)} tweets")
print(f"   Test:  {len(kaggle_texts)} tweets")
print(f"   Labels: {np.bincount(y_train)} (0=Observer, 1=Influencer)")

Loading data/train.jsonl...
   → 154914 samples, 193 columns
Loading data/kaggle_test.jsonl...
   → 103380 samples, 191 columns

📊 Résumé:
   Train: 154914 tweets
   Test:  103380 tweets
   Labels: [82674 72240] (0=Observer, 1=Influencer)


## 2. Modèle CamemBERT

In [5]:
# =============================================================================
# 🤖 CLASSE EMBEDDER
# =============================================================================

class CamemBERTEmbedder:
    """Génère des embeddings CamemBERT (multi-layer optionnel)."""
    
    def __init__(self, config: Config, device: torch.device):
        self.config = config
        self.device = device
        
        print(f"🔄 Chargement de {config.model_name}...")
        self.tokenizer = CamembertTokenizer.from_pretrained(config.model_name)
        self.model = CamembertModel.from_pretrained(
            config.model_name,
            output_hidden_states=config.use_multi_layer
        )
        self.model.to(device)
        self.model.eval()
        print(f"✅ Modèle chargé sur {device}")
    
    def _pool(self, hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Pooling des séquences."""
        if self.config.pooling == "cls":
            return hidden[:, 0, :]
        else:  # mean pooling
            mask_expanded = mask.unsqueeze(-1).float()
            sum_hidden = torch.sum(hidden * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            return sum_hidden / sum_mask
    
    @torch.no_grad()
    def embed(self, texts: List[str]) -> np.ndarray:
        """Génère les embeddings pour une liste de textes."""
        embeddings = []
        bs = self.config.batch_size
        
        for i in tqdm(range(0, len(texts), bs), desc="Embedding"):
            batch = texts[i:i+bs]
            
            # Tokenize
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.config.max_length,
                return_tensors="pt"
            )
            input_ids = encoded['input_ids'].to(self.device)
            attention_mask = encoded['attention_mask'].to(self.device)
            
            # Forward
            if self.config.use_fp16 and self.device.type == "cuda":
                with torch.cuda.amp.autocast():
                    outputs = self.model(input_ids, attention_mask=attention_mask)
            else:
                outputs = self.model(input_ids, attention_mask=attention_mask)
            
            # Extract embeddings
            if self.config.use_multi_layer:
                layers = [self._pool(outputs.hidden_states[l], attention_mask) 
                         for l in self.config.layers_to_use]
                batch_emb = torch.cat(layers, dim=-1)
            else:
                batch_emb = self._pool(outputs.last_hidden_state, attention_mask)
            
            embeddings.append(batch_emb.cpu().numpy().astype(np.float32))
            
            # Cleanup périodique
            if i % (bs * 100) == 0:
                gc.collect()
                if self.device.type == "cuda":
                    torch.cuda.empty_cache()
        
        return np.vstack(embeddings)
    
    @property
    def embedding_dim(self) -> int:
        dim = self.model.config.hidden_size
        if self.config.use_multi_layer:
            return dim * len(self.config.layers_to_use)
        return dim

# Initialiser
embedder = CamemBERTEmbedder(config, device)
print(f"\n📊 Dimension des embeddings: {embedder.embedding_dim}")

🔄 Chargement de camembert-base...
✅ Modèle chargé sur cuda

📊 Dimension des embeddings: 3072


## 3. Génération des Embeddings

⚠️ **Cette cellule peut prendre 1-2 heures sur GPU**

In [ ]:
# =============================================================================
# 🚀 GÉNÉRATION DES EMBEDDINGS
# =============================================================================

print("="*60)
print("📝 TRAIN EMBEDDINGS")
print("="*60)
X_train_emb = embedder.embed(train_texts)
print(f"   Shape: {X_train_emb.shape}")

# Sauvegarder immédiatement
np.save(os.path.join(config.embedding_dir, "X_train_embeddings.npy"), X_train_emb)
print(f"   ✅ Sauvegardé!")

# Libérer mémoire
del X_train_emb
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("📝 KAGGLE EMBEDDINGS")
print("="*60)
X_kaggle_emb = embedder.embed(kaggle_texts)
print(f"   Shape: {X_kaggle_emb.shape}")

# Sauvegarder
np.save(os.path.join(config.embedding_dir, "X_kaggle_embeddings.npy"), X_kaggle_emb)
print(f"   ✅ Sauvegardé!")

# Cleanup final
del X_kaggle_emb, embedder
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("✅ TERMINÉ!")
print("="*60)

📝 TRAIN EMBEDDINGS


Embedding:  10%|█         | 500/4842 [00:39<04:19, 16.73it/s]

In [ ]:
# =============================================================================
# 📊 VÉRIFICATION
# =============================================================================
import os

emb_dir = "./data/embeddings"
print("📁 Fichiers générés:")
for f in os.listdir(emb_dir):
    if f.endswith('.npy'):
        path = os.path.join(emb_dir, f)
        size_mb = os.path.getsize(path) / 1e6
        arr = np.load(path)
        print(f"   {f}: {arr.shape} ({size_mb:.1f} MB)")

## ✅ Prochaines étapes

1. **Features structurées** → `feature_engineering.ipynb`
2. **Modélisation** → `model_with_features.ipynb`